# 动量因子研究

`FactorParams` 提供运行输入，`FactorAnalysisParams` 继承它并增加分析设置。`FactorReportForm.build()` 构造本项目的分析参数，再通过 `params.run(Factor)` 计算固定 20 日动量并生成报告。

首个代码单元格优先加载当前项目 `src` 源码；修改源码后重跑此格，不使用环境中旧安装包。请在当前项目目录运行内核。

In [ ]:
from pathlib import Path
import importlib
import sys
import tomllib

project_root = Path.cwd().resolve()
pyproject_path = project_root / "pyproject.toml"
if not pyproject_path.is_file():
    raise FileNotFoundError("请将内核工作目录设为包含 pyproject.toml 和 src 的当前项目目录。")
with pyproject_path.open("rb") as project_file:
    project_config = tomllib.load(project_file)
package_name = (
    project_config.get("tool", {}).get("uv", {}).get("build-backend", {}).get("module-name")
    or project_config.get("project", {}).get("name", "").replace("-", "_")
)
src_path = (project_root / "src").resolve()
if not package_name or not src_path.is_dir():
    raise FileNotFoundError("当前项目缺少包名配置或 src 目录；请检查 pyproject.toml 和工作目录。")
package_path = src_path.joinpath(*package_name.split("."))
if not package_path.is_dir() and not package_path.with_suffix(".py").is_file():
    raise FileNotFoundError(f"找不到当前项目源码：{package_path}；请检查包名配置。")
sys.path[:] = [str(src_path)] + [path for path in sys.path if path != str(src_path)]
importlib.invalidate_caches()
for loaded_name in list(sys.modules):
    if loaded_name == package_name or loaded_name.startswith(package_name + "."):
        del sys.modules[loaded_name]

from uuid import uuid4

import pandas as pd
from scheme import StockPool, Universe
from factor import Factor, FactorAnalysisParams, FactorReportForm

## 报告表单

插件读取同一个 `FactorReportForm` 的 JSON Schema 生成输入框。因子没有可调算法参数。

In [ ]:
form = FactorReportForm(
    start="2026-01-01",
    end="2026-07-01",
    pool=StockPool.CSI300,
    columns=["momentum"],
    return_periods=[1, 5, 20],
    groups=5,
    n_select=2,
    weight="equal",
)
form

## 构建分析参数

`build()` 展开股票池预设。需要自定义股票池时，也可以直接创建 `FactorAnalysisParams(start=..., end=..., universe=..., columns=...)`。例如先创建 `Universe(pool=StockPool.CUSTOM, codes=["600000.SH"], derivatives={"member": {"type": "DIRECT", "op": "nullary.true", "fields": {}}}, filters=["member"])`，再用 `FactorAnalysisParams.model_validate({**dict(params), "universe": universe})` 替换股票池。自定义因子、衍生字段和 filters 也可通过 Universe 配置；不要对已经构建的股票池使用 `model_dump()` 再重建，以免预设覆盖自定义内容。

In [ ]:
params: FactorAnalysisParams = form.build()
params

## 运行完整分析

In [ ]:
report = params.run(Factor)

## 完整交互报告

In [ ]:
report.show()

## 保存全部报告（可选）

将所有原始报告数据导出为 Parquet，可供前端读取。默认不写文件；需要保存时显式设为 `True` 并运行下一格。请先将内核工作目录设为当前 Notebook 所在目录。每次导出使用独立的 `reports/<uuid>` 目录，不覆盖历史报告或 `/shared/runs`。

In [ ]:
save_report = False  # 可选：显式改为 True 后导出
if save_report:
    output = Path.cwd() / "reports" / uuid4().hex
    paths = report.save(output)
    display(pd.DataFrame({"报告": list(report.filenames), "文件": [str(path) for path in paths]}))